In [1]:
%pip install pandas pyodbc sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import pyodbc
from sqlalchemy import create_engine
from urllib.parse import quote_plus

SERVER = r"DESKTOP-PEIIU7J\SQLEXPRESS"
DATABASE = "AppLabDB"

connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

connection_url = quote_plus(connection_string)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_url}"
)

In [4]:
rejected_df = pd.read_csv("rejected_records.csv")

errors_df = pd.read_csv('error.csv')

print("Rejected records:", len(rejected_df))
print("Error logs:", len(errors_df))

display(rejected_df.head(10))
display(errors_df.head(10))

Rejected records: 6
Error logs: 7


,OrderID,CustomerID,ProductID,SaleAmount,Currency,OrderDate,RejectionReason,ErrorTimestamp,PipelineName
0,1005,NaN,PX1,89.5,USD,01/07/2023,MISSING_CUSTOMER_ID|PRODUCT_NOT_FOUND,2026-11-13T10:14:32.498Z,AppLab_Sales_ETL
1,1004,C102,P99,150.0,USD,NaN,MISSING_ORDER_DATE,2026-11-13T10:14:32.498Z,AppLab_Sales_ETL
2,1002,C101,P72,NaN,EUR,01/05/2023,MISSING_SALE_AMOUNT,2026-11-13T10:14:32.498Z,AppLab_Sales_ETL
3,1003,C100,P50,-10.0,GBP,01-06-2023,NEGATIVE_SALE_AMOUNT,2026-11-13T10:14:32.498Z,AppLab_Sales_ETL
4,1006,C101,P72,200.0,EUR,2023-13-01,INVALID_ORDER_DATE,2026-11-13T10:14:32.498Z,AppLab_Sales_ETL
5,1015,C113,P99,-25.0,USD,05/07/2023,NEGATIVE_SALE_AMOUNT,2026-11-13T10:14:32.498Z,AppLab_Sales_ETL


,OrderID,CustomerID,ProductID,ErrorType,ErrorTimestamp,PipelineName
0,1005,NaN,PX1,MISSING_CUSTOMER_ID,2026-11-13T10:14:34.450Z,AppLab_Sales_ETL
1,1005,NaN,PX1,PRODUCT_NOT_FOUND,2026-11-13T10:14:34.450Z,AppLab_Sales_ETL
2,1004,C102,P99,MISSING_ORDER_DATE,2026-11-13T10:14:34.450Z,AppLab_Sales_ETL
3,1002,C101,P72,MISSING_SALE_AMOUNT,2026-11-13T10:14:34.450Z,AppLab_Sales_ETL
4,1003,C100,P50,NEGATIVE_SALE_AMOUNT,2026-11-13T10:14:34.450Z,AppLab_Sales_ETL
5,1006,C101,P72,INVALID_ORDER_DATE,2026-11-13T10:14:34.450Z,AppLab_Sales_ETL
6,1015,C113,P99,NEGATIVE_SALE_AMOUNT,2026-11-13T10:14:34.450Z,AppLab_Sales_ETL


In [5]:
import pyodbc
import pandas as pd
import numpy as np

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    r"SERVER=DESKTOP-PEIIU7J\SQLEXPRESS;"
    "DATABASE=AppLabDB;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

cursor = conn.cursor()

print("Connected successfully!")

Connected successfully!


In [6]:
cursor.execute("""
    SELECT
        (SELECT COUNT(*) FROM dbo.RejectedRecords) AS RejectedCount,
        (SELECT COUNT(*) FROM dbo.ErrorLogs) AS ErrorCount
""")

print(cursor.fetchone())

(0, 0)


In [7]:
# Work on copies
rejected_load = rejected_df.copy()
errors_load = errors_df.copy()

# Convert missing values to None
rejected_load = rejected_load.astype(object).where(
    pd.notna(rejected_load), None
)

errors_load = errors_load.astype(object).where(
    pd.notna(errors_load), None
)

# Convert timestamps to timezone-naive UTC datetimes
for df in [rejected_load, errors_load]:
    df["ErrorTimestamp"] = (
        pd.to_datetime(df["ErrorTimestamp"], utc=True)
        .dt.tz_localize(None)
    )

print(rejected_load.dtypes)
print(errors_load.dtypes)

OrderID                    object
CustomerID                 object
ProductID                  object
SaleAmount                 object
Currency                   object
OrderDate                  object
RejectionReason            object
ErrorTimestamp     datetime64[ns]
PipelineName               object
dtype: object
OrderID                   object
CustomerID                object
ProductID                 object
ErrorType                 object
ErrorTimestamp    datetime64[ns]
PipelineName              object
dtype: object


In [8]:
rejected_sql = """
INSERT INTO dbo.RejectedRecords
(
    OrderID,
    CustomerID,
    ProductID,
    SaleAmount,
    Currency,
    OrderDate,
    RejectionReason,
    ErrorTimestamp,
    PipelineName
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
"""

rejected_rows = list(
    rejected_load[
        [
            "OrderID",
            "CustomerID",
            "ProductID",
            "SaleAmount",
            "Currency",
            "OrderDate",
            "RejectionReason",
            "ErrorTimestamp",
            "PipelineName"
        ]
    ].itertuples(index=False, name=None)
)

cursor.executemany(rejected_sql, rejected_rows)

print(f"Inserted {len(rejected_rows)} rejected records")

Inserted 6 rejected records


In [9]:
errors_sql = """
INSERT INTO dbo.ErrorLogs
(
    OrderID,
    CustomerID,
    ProductID,
    ErrorType,
    ErrorTimestamp,
    PipelineName
)
VALUES (?, ?, ?, ?, ?, ?)
"""

error_rows = list(
    errors_load[
        [
            "OrderID",
            "CustomerID",
            "ProductID",
            "ErrorType",
            "ErrorTimestamp",
            "PipelineName"
        ]
    ].itertuples(index=False, name=None)
)

cursor.executemany(errors_sql, error_rows)

print(f"Inserted {len(error_rows)} error log records")

Inserted 7 error log records


In [10]:
conn.commit()

print("Both tables loaded successfully!")

Both tables loaded successfully!


In [11]:
verification_query = """
SELECT 'RejectedRecords' AS TableName, COUNT(*) AS TotalRows
FROM dbo.RejectedRecords

UNION ALL

SELECT 'ErrorLogs', COUNT(*)
FROM dbo.ErrorLogs;
"""

verification_df = pd.read_sql(
    verification_query,
    conn
)

display(verification_df)

C:\Users\hp\AppData\Local\Temp\ipykernel_18716\3752875557.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  verification_df = pd.read_sql(


,TableName,TotalRows
0,RejectedRecords,6
1,ErrorLogs,7
